# Plot raw detector frames

Inspect raw detector data straight off disk, no integration. Useful for:

- Quick sanity check of a fresh dataset before doing heavier analysis.
- Spotting saturated pixels, beamstop shadows, weird artefacts.
- Picking representative frames for follow-up workflows.

In [ ]:
import matplotlib.pyplot as plt

from giwaxs_analysis import calibration, io, plotting

## 1. (Optional) Load a calibration

Loading a `Calibration` gives `plot_detector` access to the mask, so masked pixels (dead pixels, module gaps, beamstop shadow) are rendered transparent instead of skewing the colour scale.

You can skip this cell entirely and pass `calib=None` below — you'll just see the raw data with the masked regions still visible.

In [ ]:
PONI_PATH = '../data/calibrant_AgBh.poni'
MASK_PATH = '../data/mask.edf'

calib = calibration.load_calibration(PONI_PATH, MASK_PATH)

## 2. Plot a single frame

Log scale and viridis colormap by default; tweak `VMIN`/`VMAX` to bring out faint features or saturate bright ones.

In [ ]:
FRAME_PATH = '../data/insitu/in_situ_GIWAXS_1.edf'

VMIN = None     # e.g. 10 to suppress noise
VMAX = None     # e.g. 5000 to saturate bright pixels and bring out weak features
LOG_SCALE = True  # ← False for linear colour scale

frame = io.load_frame(FRAME_PATH)
fig, ax = plotting.plot_detector(
    frame,
    calib=calib,
    log=LOG_SCALE,
    vmin=VMIN,
    vmax=VMAX,
    title=FRAME_PATH.rsplit('/', 1)[-1],
)
plt.show()

## 3. Plot every frame in a folder

`list_frames` + `load_stack` pull every detector file in a folder into a single 3D array, then `plot_detector_grid` draws them all sharing the same colour scale (so frame-to-frame intensity differences are real).

In [ ]:
FOLDER = '../data/insitu/'

paths = io.list_frames(FOLDER)
print(f'Found {len(paths)} frames in {FOLDER}')

stack = io.load_stack(paths)
labels = [p.stem for p in paths]

fig, axes = plotting.plot_detector_grid(
    stack,
    calib=calib,
    labels=labels,
    ncols=3,
    log=True,
    vmin=None,
    vmax=None,
)
plt.show()

## A note on file formats

`io.list_frames` defaults to matching `*.edf` (ESRF / ALS convention). For Diamond data (TIFF), pass an explicit pattern:

```python
io.list_frames(FOLDER, pattern='*.tif')
```